# BITS Pilani - M.Tech (AIML) Machine Learning Assignment 2
## Classification Modeling & Comparative Evaluation Pipeline
**Dataset**: UCI Default of Credit Card Clients Dataset  
**Target**: Default Payment Next Month (0 = No Default, 1 = Default)  
**Features**: 23 demographic, repayment status, bill statement, and prior payment history variables.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

### 1. Dataset Ingestion & Exploration

In [2]:
raw_file = os.path.join("..", "data_raw", "default of credit card clients.xls")
if not os.path.exists(raw_file):
    raw_file = os.path.join("data_raw", "default of credit card clients.xls")
df = pd.read_excel(raw_file, header=1, engine="calamine")
if "ID" in df.columns:
    df = df.drop(columns=["ID"])
target_col = "default payment next month"
df = df.rename(columns={target_col: "default"})
print(f"Dataset Shape: {df.shape}")
df.head()

### 2. Stratified Train / Test Split (80 / 20)

In [3]:
X = df.drop(columns=["default"])
y = df["default"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train default rate: {y_train.mean():.4f}, y_test default rate: {y_test.mean():.4f}")

### 3. Model Training & Evaluation (5 Classifiers)

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_split=10, min_samples_leaf=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1),
    "Naive Bayes": GaussianNB(var_smoothing=1e-8),
    "Support Vector Machine": SVC(kernel="rbf", C=1.0, random_state=42)
}

results = []
for name, clf in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    if hasattr(clf, "predict_proba"):
        y_scores = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_scores = pipe.decision_function(X_test)
    else:
        y_scores = y_pred
    
    results.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_scores), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
display(results_df)